# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR²) Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library. The dataset is defined by a Croissant schema and accessible via a public schema URL.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In `mlcroissant`, datasets contain one or more *record sets*, each with its unique `@id` as defined in the schema. Each record set contains fields and columns, also referenced via their `@id`s.

In [ ]:
# List available record sets by their @id, name, and fields

record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found in the schema.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        print(f"  Name: {rs.get('name', '[no name]')}")
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        print(f"  Fields: {[f['@id'] for f in fields] if fields else '[none]'}")
        print('-'*60)

Below, we demonstrate how to inspect the first few records of the main record set. Replace the `record_set_id` variable value with the appropriate `@id` from the listing above.

In [ ]:
# Choose the main record set (replace with the actual @id from above if needed)
record_set_id = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json#records_clinical'

# If unsure, list all available @ids again
available_ids = [rs['@id'] for rs in dataset.record_sets]
print('Available RecordSet @ids:', available_ids)

# Print 3 example records for the selected record set
try:
    for i, record in enumerate(dataset.records(record_set=record_set_id)):
        if i >= 3:
            break
        print(f"Record {i+1}: {record}")
except Exception as e:
    print(f"Error fetching records: {e}")

## 3. Data Extraction
Load data from the main record set(s) into DataFrames for analysis. Use the record set and field `@id`s found earlier.

We'll extract all defined record sets into pandas DataFrames, referenced by their `@id`.

In [ ]:

# Get all record set @ids (edit or subset as desired)
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for rsid in record_set_ids:
    try:
        records = list(dataset.records(record_set=rsid))
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        print(f'Loaded {len(df)} records for RecordSet: {rsid}')
    except Exception as e:
        print(f'Error loading {rsid}:', e)

# Show columns and preview of the main clinical record set (update below if needed)
main_rs = record_set_ids[0] if record_set_ids else None
if main_rs and main_rs in dataframes:
    print(f"Columns in RecordSet {main_rs}: ", dataframes[main_rs].columns.tolist())
    display(dataframes[main_rs].head())
else:
    print("No dataframes available.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, we filter the dataset based on a chosen numeric field. Replace `<numeric_field_id>` and `<group_field_id>` with the appropriate `@id` of the column to use (see DataFrame columns above for options).

Example: We'll use the '@id' of a field such as 'Age' or a numeric interval.

In [ ]:
# Replace these with real column @ids as appropriate
# For illustration, suppose the field @id is 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json#patient_age' (change as needed!)
df = dataframes[main_rs]

numeric_field = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json#age'  # Example only; update to actual field
if numeric_field not in df.columns:
    print(f"Field {numeric_field} not found in DataFrame columns:")
    print(df.columns.tolist())
else:
    # Filter by a threshold, e.g. age > 50
    threshold = 50
    filtered_df = df[df[numeric_field].astype(float) > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalization (z-score) of the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()) / filtered_df[numeric_field].astype(float).std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouping by another field (e.g., sex or cancer_type)
    group_field = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json#sex'  # Update to suitable field
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped average {numeric_field} by {group_field}:")
        display(grouped_df)
    else:
        print(f"Group field {group_field} not in columns.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We plot a histogram for our chosen numeric field and a boxplot by group (if grouping is available).

In [ ]:
# Only plot if columns are available
if numeric_field in df.columns:
    # Histogram
    plt.figure(figsize=(6, 4))
    df[numeric_field].astype(float).hist(bins=15)
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.title(f"Distribution of {numeric_field}")
    plt.show()

    # Boxplot by group if available
    if group_field in df.columns:
        plt.figure(figsize=(8, 5))
        df.boxplot(column=numeric_field, by=group_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.suptitle("")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print(f"Cannot plot: numeric_field '{numeric_field}' not found in columns.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we demonstrated loading the FAIR² dataset via its Croissant schema, listing its available record sets and fields by `@id`, and showed how to extract, process, and visualize selected fields using the `mlcroissant` library and standard Python data tools.

- To repeat or extend this analysis: swap the `@id`s above for others listed in your schema or DataFrame columns.
- All data elements are referenced by `@id` for reproducibility and traceability, consistent with Croissant and `mlcroissant` data modeling best practices.

Explore further using more complex filters, analyses, or by combining other record sets as needed!